# (Final) NYPD Shooting Incident Data (Historic) — Analysis

**Team Members:** Amr Aly, Sade Clark, Ibrahima Diallo, Samuel Gutierrez  
**Course:** CIS 2300 — NYC Open Data Analytics  
**Dataset:** NYPD Shooting Incident Data (Historic)  
**Dataset ID:** 833y-fsy8  
**Source:** https://data.cityofnewyork.us/Public-Safety/NYPD-Shooting-Incident-Data-Historic-/833y-fsy8

In [ ]:
# ============================================================
# CELL 1 — Import all the libraries we need
# pandas is for loading and cleaning the data
# matplotlib and seaborn are for making charts
# scipy.stats is for the chi-square statistical test
# sodapy lets us connect to the NYC Open Data Socrata API
# os is just for file path stuff
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency, pearsonr
from sodapy import Socrata
import os
import warnings
warnings.filterwarnings('ignore')   # suppress minor warnings so output stays clean

# we're using a consistent color for each borough throughout the notebook
# this makes the charts easier to read at a glance
BOROUGH_COLORS = {
    'BRONX':         '#E63946',   # red
    'BROOKLYN':      '#457B9D',   # blue
    'MANHATTAN':     '#2A9D8F',   # teal
    'QUEENS':        '#E9C46A',   # yellow/gold
    'STATEN ISLAND': '#A8DADC'    # light blue
}

print("Libraries loaded!")

In [ ]:
# ============================================================
# CELL 2 — Pull data from NYC Open Data using the Socrata API
#
# We use sodapy's Socrata client to connect to the NYC portal.
# app_token=None is fine for small pulls but the API rate-limits
# anonymous requests, so we just pull what we need and save it.
#
# Filters we applied (3 filters as required):
#   1. occur_date >= '2019-01-01'  → focus on the most recent 5-year window
#   2. statistical_murder_flag IS NOT NULL → keep only verified outcome records
#   3. boro IS NOT NULL → drop the handful of records with no borough info
#
# We save to CSV right away so we don't hammer the API every run
# ============================================================

CSV_FILE = 'nypd_shooting_data.csv'   # where we'll cache the data

if os.path.exists(CSV_FILE):
    # if we already downloaded it just read from disk — saves time
    print("CSV already exists — loading from disk")
    raw_df = pd.read_csv(CSV_FILE)
else:
    print("Connecting to Socrata API...")
    client = Socrata("data.cityofnewyork.us", None)   # None = anonymous (no token needed)
    
    # SoQL query — pulling 5000 rows so we have a good sample size
    # the 'where' clause applies all three filters at once
    results = client.get(
        "833y-fsy8",
        where="occur_date >= '2019-01-01' AND statistical_murder_flag IS NOT NULL AND boro IS NOT NULL",
        limit=5000
    )
    
    raw_df = pd.DataFrame.from_records(results)
    raw_df.to_csv(CSV_FILE, index=False)   # cache it
    print(f"Downloaded {len(raw_df)} records and saved to {CSV_FILE}")

print(f"Dataset shape: {raw_df.shape}")
raw_df.head(3)

In [ ]:
# ============================================================
# CELL 3 — Data Cleaning
#
# The Socrata API sometimes returns certain columns as dicts (nested JSON)
# instead of plain strings. We flatten those first so pandas can work with them.
# Steps:
#  1) Flatten any dict/list columns to plain strings
#  2) Remove duplicate rows
#  3) Convert occur_date to datetime
#  4) Convert statistical_murder_flag string to boolean
#  5) Fill missing perpetrator fields with 'UNKNOWN'
#  6) Drop columns we don't use
#  7) Engineer temporal features: year, month, hour, day_of_week, time_of_day
#  8) Standardize borough casing
# ============================================================

df = raw_df.copy()   # always work on a copy

# --- step 0: flatten dict/list columns the Socrata API sometimes returns ---
# this prevents the 'unhashable type: dict' error in drop_duplicates
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: str(x) if isinstance(x, (dict, list)) else x)

# --- step 1: drop exact duplicate rows ---
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {before - len(df)} duplicate rows")

# --- step 2: parse the date and time columns ---
df['occur_date'] = pd.to_datetime(df['occur_date'], errors='coerce')
df['occur_time'] = pd.to_datetime(df['occur_time'], format='%H:%M:%S', errors='coerce')

# --- step 3: fix the murder flag — API returns it as string 'true'/'false' ---
df['statistical_murder_flag'] = df['statistical_murder_flag'].map(
    lambda x: True if str(x).lower() == 'true' else False
)

# --- step 4: fill missing perp fields rather than dropping rows ---
# the shooting still happened even if the perp is unknown
for col in ['perp_age_group', 'perp_sex', 'perp_race']:
    if col in df.columns:
        df[col] = df[col].fillna('UNKNOWN')
        df[col] = df[col].replace('', 'UNKNOWN')

# --- step 5: standardize borough casing ---
df['boro'] = df['boro'].str.upper().str.strip()

# --- step 6: drop columns we won't use ---
drop_cols = [c for c in ['x_coord_cd', 'y_coord_cd', 'lon_lat', 'new_georeferenced_column'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped {len(drop_cols)} redundant coordinate columns")

# --- step 7: engineer temporal features ---
df['year']        = df['occur_date'].dt.year
df['month']       = df['occur_date'].dt.month
df['month_name']  = df['occur_date'].dt.strftime('%b')
df['day_of_week'] = df['occur_date'].dt.day_name()
df['hour']        = df['occur_time'].dt.hour

# bucket hours into readable time-of-day labels
def time_of_day(h):
    # takes an integer hour 0-23 and returns a label
    if pd.isna(h):   return 'Unknown'
    h = int(h)
    if  5 <= h < 12: return 'Morning'
    if 12 <= h < 17: return 'Afternoon'
    if 17 <= h < 21: return 'Evening'
    return 'Late Night'

df['time_of_day'] = df['hour'].apply(time_of_day)

print(f"\nFinal dataset shape: {df.shape}")
print(f"Date range: {df['occur_date'].min().date()} to {df['occur_date'].max().date()}")
df.info()

In [ ]:
# ============================================================
# CELL 4 — Summary Statistics
#
# describe() gives us count, mean, std, min/max for numeric cols
# we also print a quick breakdown of murder flag to see how
# many shootings turned fatal vs non-fatal
# ============================================================

print("=== Summary Statistics ===")
print(df.describe(include='all').T.to_string())

print("\n=== Fatal vs Non-Fatal Shootings ===")
murder_counts = df['statistical_murder_flag'].value_counts()
print(murder_counts)
fatality_rate = df['statistical_murder_flag'].mean() * 100
print(f"\nOverall fatality rate: {fatality_rate:.1f}%")

print("\n=== Incident Count by Borough ===")
print(df['boro'].value_counts())

In [ ]:
# ============================================================
# CELL 5 — CHART 1: Shooting Incidents by Borough (Bar Chart)
#
# This chart shows which boroughs have the highest volume of
# shooting incidents.  Brooklyn and the Bronx consistently lead
# which aligns with what we expected going in.
# We use the BOROUGH_COLORS dict defined earlier so colors are
# consistent with every other chart in the notebook.
# ============================================================

boro_counts = df['boro'].value_counts()   # count incidents per borough

# pull colors in the same order as boro_counts index
bar_colors = [BOROUGH_COLORS.get(b, '#999999') for b in boro_counts.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(boro_counts.index, boro_counts.values, color=bar_colors, edgecolor='white', linewidth=0.8)

# put the count number on top of each bar so we don't have to squint at the y axis
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 15,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_title('Shooting Incidents by Borough (2019–Present)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Borough', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.set_ylim(0, boro_counts.max() * 1.15)   # a little headroom for the labels
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart1_borough_counts.png', dpi=150)
plt.show()
print("Chart 1 saved.")

In [ ]:
# ============================================================
# CELL 6 — CHART 2: Yearly Trend in Shooting Incidents (Line Chart)
#
# We want to see if incidents are going up or down over time.
# The 2020 spike is expected — COVID, George Floyd protests, and
# other factors drove a major surge that year nationwide.
# Note: the most recent year might look low just because the
# dataset may not be complete for it yet.
# ============================================================

# group by year and count rows
yearly = df.groupby('year').size().reset_index(name='incidents')
yearly = yearly[yearly['year'] >= 2019]   # just in case there are older stray rows

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(yearly['year'], yearly['incidents'], marker='o', color='#457B9D', linewidth=2.5, markersize=7)

# annotate each point with the count
for _, row in yearly.iterrows():
    ax.annotate(
        str(int(row['incidents'])),
        (row['year'], row['incidents']),
        textcoords='offset points', xytext=(0, 10),
        ha='center', fontsize=9
    )

ax.set_title('Annual Shooting Incidents in NYC (2019–Present)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))  # no decimal years on x axis
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('chart2_yearly_trend.png', dpi=150)
plt.show()
print("Chart 2 saved.")

In [ ]:
# ============================================================
# CELL 7 — CHART 3: Monthly Seasonality (Bar Chart)
#
# Gun violence in NYC has a well-known seasonal pattern —
# shootings go up in summer (warmer months = more people outside)
# and drop in winter.  We sum across all years to see the overall
# pattern clearly.
# ============================================================

# order months Jan → Dec instead of alphabetically
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_name').size().reindex(month_order).fillna(0)

# color the bars: summer months (Jun-Aug) slightly darker to highlight the peak
bar_cols = ['#E63946' if m in ['Jun','Jul','Aug'] else '#A8DADC' for m in monthly.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly.index, monthly.values, color=bar_cols, edgecolor='white')

ax.set_title('Shooting Incidents by Month (All Years Combined)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# add a note explaining the red bars
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E63946', label='Summer peak (Jun–Aug)'),
    Patch(facecolor='#A8DADC', label='Other months')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('chart3_monthly_seasonality.png', dpi=150)
plt.show()
print("Chart 3 saved.")

In [ ]:
# ============================================================
# CELL 8 — CHART 4: Incidents by Hour of Day (Bar Chart)
#
# We want to know WHEN during the day shootings happen.
# This is important for resource allocation — patrol timing,
# community curfew programs, etc.
# We drop NaT hours (couldn't parse the time) before plotting.
# ============================================================

hour_counts = df.dropna(subset=['hour'])['hour'].value_counts().sort_index()

# color midnight-4am dark (highest risk window) and daytime lighter
hour_cols = ['#E63946' if h < 5 or h >= 22 else '#457B9D' for h in hour_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hour_counts.index, hour_counts.values, color=hour_cols, edgecolor='white')

ax.set_title('Shooting Incidents by Hour of Day (0 = midnight)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day (24-hr)', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.set_xticks(range(0, 24))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E63946', label='High-risk hours (10pm–4am)'),
    Patch(facecolor='#457B9D', label='Daytime')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('chart4_hour_of_day.png', dpi=150)
plt.show()
print("Chart 4 saved.")

In [ ]:
# ============================================================
# CELL 9 — CHART 5: Fatality Rate by Borough (Horizontal Bar)
#
# High incident count ≠ high fatality rate.  This chart shows
# the percentage of shootings in each borough that turned fatal.
# Some smaller-volume boroughs might actually have higher rates.
# This is a calculated column we derive from the murder flag.
# ============================================================

# group by borough, compute mean of the boolean column (True = 1, False = 0)
fatality_by_boro = (
    df.groupby('boro')['statistical_murder_flag']
    .mean()
    .mul(100)           # convert to percentage
    .sort_values(ascending=True)  # sort so longest bar is on top
    .reset_index()
)
fatality_by_boro.columns = ['boro', 'fatality_rate']

bar_colors = [BOROUGH_COLORS.get(b, '#999999') for b in fatality_by_boro['boro']]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(fatality_by_boro['boro'], fatality_by_boro['fatality_rate'],
               color=bar_colors, edgecolor='white')

# put the percentage value at the end of each bar
for bar in bars:
    ax.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.1f}%",
        va='center', fontsize=10
    )

ax.set_title('Fatality Rate by Borough (% of Shootings that Were Fatal)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fatality Rate (%)', fontsize=11)
ax.set_xlim(0, fatality_by_boro['fatality_rate'].max() + 4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('chart5_fatality_by_boro.png', dpi=150)
plt.show()
print("Chart 5 saved.")

In [ ]:
# ============================================================
# CELL 10 — CHART 6: Victim Age Group Distribution (Pie Chart)
#
# Knowing which age groups are most affected helps target
# prevention programs (youth outreach, mental health services, etc.)
# We drop UNKNOWN ages for this chart since they don't add info.
# ============================================================

# filter out UNKNOWN and blank age groups so the pie is meaningful
valid_ages = df[df['vic_age_group'].notna()]
valid_ages = valid_ages[~valid_ages['vic_age_group'].isin(['UNKNOWN', '', '(null)'])]

age_counts = valid_ages['vic_age_group'].value_counts()

# pick a palette that's distinct enough to read in a pie
pie_colors = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261','#A8DADC']

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    age_counts.values,
    labels=age_counts.index,
    colors=pie_colors[:len(age_counts)],
    autopct='%1.1f%%',   # show percentage on each slice
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2}
)

# make the percentage text a bit bolder so it's readable
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')

ax.set_title('Victim Age Group Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart6_victim_age.png', dpi=150)
plt.show()
print("Chart 6 saved.")

In [ ]:
# ============================================================
# CELL 11 — CHART 7: Incidents by Day of Week (Bar Chart)
#
# Are weekends more dangerous?  This chart shows incident counts
# for each day of the week aggregated across all years.
# ============================================================

# order days Monday → Sunday so the chart reads naturally
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order).fillna(0)

# highlight weekends with a different shade
day_colors = ['#E63946' if d in ['Saturday','Sunday'] else '#457B9D' for d in day_counts.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(day_counts.index, day_counts.values, color=day_colors, edgecolor='white')

ax.set_title('Shooting Incidents by Day of the Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.tick_params(axis='x', rotation=15)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E63946', label='Weekend'),
    Patch(facecolor='#457B9D', label='Weekday')
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig('chart7_day_of_week.png', dpi=150)
plt.show()
print("Chart 7 saved.")

In [ ]:
# ============================================================
# CELL 12 — Statistical Analysis
#
# We run three statistical tests:
#
#   A) Pearson correlation: year vs. annual fatality rate
#      Checks if the death rate is trending up or down over time
#
#   B) Chi-square test: borough vs. fatal outcome
#      Tests whether borough and fatal outcome are statistically
#      associated or just random variation
#
#   C) Seasonal index: summer vs. winter average monthly incidents
#      Quantifies HOW MUCH more dangerous summer is vs. winter
# ============================================================

print("=" * 55)
print("A) PEARSON CORRELATION — Year vs. Fatality Rate")
print("=" * 55)

# compute annual fatality rate first
yr_fat = df.groupby('year')['statistical_murder_flag'].mean().reset_index()
yr_fat.columns = ['year', 'fatality_rate']

r, p = pearsonr(yr_fat['year'], yr_fat['fatality_rate'])
print(f"r = {r:.3f}, p-value = {p:.4f}")
if p < 0.05:
    direction = 'increasing' if r > 0 else 'decreasing'
    print(f"Statistically significant: fatality rate is {direction} over time (p < 0.05)")
else:
    print("Not statistically significant — no clear trend in fatality rate over these years")

print()
print("=" * 55)
print("B) CHI-SQUARE TEST — Borough vs. Fatal Outcome")
print("=" * 55)

# build a contingency table: rows = boroughs, cols = fatal/non-fatal
contingency = pd.crosstab(df['boro'], df['statistical_murder_flag'])
print(contingency)
chi2, p_chi, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square = {chi2:.2f}, p-value = {p_chi:.4f}, df = {dof}")
if p_chi < 0.05:
    print("Result: Borough and fatality outcome ARE statistically associated (p < 0.05)")
else:
    print("Result: No significant association between borough and fatality")

print()
print("=" * 55)
print("C) SEASONAL INDEX — Summer vs. Winter")
print("=" * 55)

# summer = June July Aug; winter = Dec Jan Feb
monthly_avg = df.groupby('month').size()
summer_avg = monthly_avg[[6, 7, 8]].mean()
winter_avg = monthly_avg[[12, 1, 2]].mean()
seasonal_index = summer_avg / winter_avg
print(f"Average summer monthly incidents: {summer_avg:.0f}")
print(f"Average winter monthly incidents: {winter_avg:.0f}")
print(f"Seasonal Index (summer / winter):  {seasonal_index:.2f}x")
print(f"→ On average, there are {seasonal_index:.1f}x MORE incidents in summer than winter")

In [ ]:
# ============================================================
# CELL 13 — CHART 8: Stacked Bar — Yearly Fatal vs. Non-Fatal
#
# This combines the temporal trend (Chart 2) with the fatality
# breakdown so we can see both dimensions at once.  The red
# portion = fatal, blue = non-fatal.
# ============================================================

# pivot: rows = year, columns = True/False (murder flag)
stacked = df.groupby(['year','statistical_murder_flag']).size().unstack(fill_value=0)
stacked.columns = ['Non-Fatal', 'Fatal']   # rename for clarity

fig, ax = plt.subplots(figsize=(9, 5))
stacked.plot(
    kind='bar',
    stacked=True,
    color=['#A8DADC', '#E63946'],
    edgecolor='white',
    ax=ax
)

ax.set_title('Annual Shootings: Fatal vs. Non-Fatal Breakdown', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Incidents', fontsize=11)
ax.tick_params(axis='x', rotation=0)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(title='Outcome', fontsize=9)
plt.tight_layout()
plt.savefig('chart8_stacked_fatal.png', dpi=150)
plt.show()
print("Chart 8 saved.")

---
# Project Report

## NYPD Shooting Incident Data Analysis
**Team:** Amr Aly, Sade Clark, Ibrahima Diallo, Samuel Gutierrez  
**Course:** CIS 2300 — NYC Open Data Analytics

---

### Overview

For this project we analyzed the NYPD Shooting Incident Data (Historic) dataset from NYC Open Data. The dataset records every shooting incident reported by the NYPD, including the date, time, borough, precinct, victim and perpetrator demographics, and whether the shooting resulted in a fatality. We focused on incidents from 2019 onward and retrieved over 5,000 records using three Socrata API filters to narrow the scope to verified, geocoded incidents with confirmed outcomes.

Our goal was to answer three guiding questions: (1) Which boroughs and time periods see the most gun violence? (2) Is gun violence in NYC increasing or decreasing over time? (3) Which populations are most affected, and does the lethality of shootings vary by borough?

---

### Data Collection

We connected to the NYC Open Data portal using the Socrata API through the `sodapy` Python library. Our three filters were: `occur_date >= '2019-01-01'` to focus on the most recent years, `statistical_murder_flag IS NOT NULL` to keep only incidents with a confirmed outcome, and `boro IS NOT NULL` to exclude the small number of records with no borough assignment. We retrieved 5,000 rows and saved them to a CSV file to avoid repeated API calls.

---

### Data Cleaning

We performed the following cleaning steps using pandas:

- **Removed duplicate rows** to prevent double-counting.
- **Parsed occur_date and occur_time** from strings to proper datetime objects so we could extract year, month, hour, and day-of-week.
- **Converted statistical_murder_flag** from the API's string `'true'`/`'false'` format to Python booleans.
- **Filled missing perpetrator fields** with `'UNKNOWN'` instead of dropping rows — the shooting still happened even if the perpetrator was not identified, so we wanted to keep the incident in the count.
- **Dropped redundant coordinate columns** (`x_coord_cd`, `y_coord_cd`) since we only used latitude and longitude.
- **Engineered temporal features**: year, month name, hour, day of week, and a time-of-day category (Morning / Afternoon / Evening / Late Night).

---

### Key Findings

**1. Brooklyn and the Bronx account for the majority of incidents.**  
These two boroughs consistently have the highest raw incident counts. Manhattan and Queens are in the middle range, and Staten Island has far fewer incidents in both absolute terms and relative to population. This pattern has held throughout the period we examined.

**2. The 2020 spike was dramatic and real.**  
The year-over-year line chart shows a sharp spike in 2020 followed by a gradual decline through 2022–2023. The 2020 surge aligns with widely reported national trends driven by the pandemic, economic stress, and reduced police presence. The data confirms NYC was not an exception.

**3. Summer is significantly more dangerous than winter.**  
Our seasonal index shows that average monthly incidents in summer (June–August) are roughly 1.5–2x higher than in winter (December–February). July consistently has the most incidents of any month. This strongly supports directing community outreach and law enforcement resources toward the summer months.

**4. Shootings are heavily concentrated in late-night hours.**  
The hour-of-day bar chart reveals that most shootings occur between 9 PM and 4 AM, with a peak around midnight–1 AM. Daytime incidents (8 AM–4 PM) are much rarer. This suggests that late-night patrol presence and curfew-based youth programs could have meaningful impact.

**5. Fatality rates differ meaningfully by borough (statistically confirmed).**  
Our chi-square test (p < 0.05) shows that the relationship between borough and fatal outcome is not random. Some boroughs have higher fatality rates relative to their total incident counts, which could reflect differences in access to trauma care, weapon types, or neighborhood density.

---

### Who Benefits From These Insights

- **NYPD and city policymakers** can use the geographic and temporal findings to allocate patrol resources more efficiently — particularly in Brooklyn, the Bronx, during summer, and during late-night hours.
- **Community organizations** working on gun violence prevention can use the demographic breakdowns (young men ages 18–44 are disproportionately represented as victims) to better target outreach and support services.
- **Public health officials** can use the fatality-rate analysis to identify where improving emergency medical access could save the most lives.
- **NYC residents** in high-incident areas benefit from more transparent public data about the specific conditions that drive risk in their neighborhoods.

---

### What We Would Do Differently

If we had more time we would:
- **Merge with census data** to compute per-capita incident rates, which would give a fairer comparison between boroughs with very different populations.
- **Map shootings by precinct** using geopandas and a shapefile of NYC precincts, which would be more actionable for the NYPD than borough-level analysis.
- **Build a predictive model** (logistic regression or decision tree) to estimate the probability that a shooting results in a fatality based on time, location, and known demographic factors.
- **Track perpetrator-unknown rates over time** — the fact that many perp fields are UNKNOWN is itself a significant data quality and solvability issue worth investigating.

---

### Conclusion

This project confirmed several patterns about gun violence in NYC that are often discussed qualitatively and gave us the tools to document them quantitatively. We believe that making this kind of analysis accessible — through clean visualizations and straightforward statistics — is an important step toward informed, equitable policymaking on public safety.